In [1]:
import os
from tqdm import tqdm
from copy import deepcopy

import torch
from torch.utils import benchmark
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageNet
from torchmetrics import Accuracy

from dinov3.hub.classifiers import dinov3_vit7b16_lc

import tome
import spatial_tome

In [2]:
REPO_DIR = "/home/wiss/schnaus/projects/dinov3"
weights_dir = "/storage/user/schnaus/dinov3/"
device = torch.device("cuda")
dtype = torch.bfloat16

In [3]:
# From DINOv3
def make_transform(resize_size: int | list[int] = 768):
    to_tensor = transforms.ToTensor()
    resize = transforms.Resize((resize_size, resize_size), antialias=True)
    normalize = transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )
    return transforms.Compose([to_tensor, resize, normalize])

In [4]:
# Load the model directly with bfloat16 weights
with torch.device("meta"):
    model = dinov3_vit7b16_lc(pretrained=False, device=torch.device("meta"))

weights_path = os.path.join(weights_dir, "dinov3_vit7b16_imagenet1k_linear_head-90d8ed92.pth")
weights_state_dict = torch.load(weights_path, map_location="cpu", mmap=True)
backbone_weights_path = os.path.join(weights_dir, "dinov3_vit7b16_pretrain_lvd1689m-a955f4ea.pth")
backbone_weights_state_dict = torch.load(backbone_weights_path, map_location="cpu", mmap=True)
model.backbone.load_state_dict(backbone_weights_state_dict, strict=True, assign=True)
model.linear_head.load_state_dict(weights_state_dict, strict=True, assign=True)

model = model.to(dtype).eval()

In [5]:
dataset = ImageNet(
    "/storage/group/dataset_mirrors/imagenet2012/imagenet2012_download", split="val", transform=make_transform(512)
)

In [6]:
def evaluate(model, prefix='', num_samples=128, dataset=dataset, batch_size=1, num_workers=4, device=device, dtype=dtype, verbose=False, seed=42):
    torch.manual_seed(seed)
    indices = torch.randperm(len(dataset))[:num_samples]  # use a subset for faster evaluation
    subset = Subset(dataset, indices)
    dataloader = DataLoader(subset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    accuracy = Accuracy(task="multiclass", num_classes=1000, average="macro").to(device, dtype)
    model = model.to(device, dtype).eval()
    with torch.inference_mode(), torch.autocast(device_type=device.type, dtype=dtype):
            for images, labels in tqdm(dataloader, disable=not verbose):
                images = images.to(device, dtype)
                logits = model(images)
                accuracy.update(logits, labels.to(device))
            
            with torch.profiler.profile(with_flops=True) as p, torch.nn.attention.sdpa_kernel(torch.nn.attention.SDPBackend.MATH):
                _ = model(images[:1])
            flops = sum(k.flops for k in p.key_averages())

            t = benchmark.Timer(
                stmt="model(images)",
                setup="",
                globals={"model": model, "images": images[:1]},
                num_threads=1,
            )
            time = t.timeit(10)

    accuracy_value = accuracy.compute().item()
    print(f"{prefix:<25}Accuracy: {accuracy_value:2.2%},\tFLOPs: {flops / 1e9:7.1f} GFLOPs,\tTime: {time.mean * 1000:5.1f} ms")

In [7]:
num_samples = 1024
batch_size = 32

# No ToMe
model_local = deepcopy(model)
evaluate(model_local, num_samples=num_samples, prefix='No ToMe:', batch_size=batch_size)
del model_local
torch.cuda.empty_cache()

# ToMe
for r in [4, 8, 16, 32]:
    model_local = deepcopy(model)
    tome.patch.dinov3(model_local, trace_source=False, prop_attn=True)
    model_local.r = r
    evaluate(model_local, num_samples=num_samples, prefix=f'ToMe (r={r}):', batch_size=batch_size)
    del model_local
    torch.cuda.empty_cache()

# ToMe before attention
for r in [4, 8, 16, 32]:
    model_local = deepcopy(model)
    tome.patch_pre.dinov3(model_local, trace_source=False, prop_attn=True)
    model_local.r = r
    evaluate(model_local, num_samples=num_samples, prefix=f'ToMe Pre (r={r}):', batch_size=batch_size)
    del model_local
    torch.cuda.empty_cache()

# ToMe before attention with different r for q and k
for r in [(32, 32), (8, 64), (16, 64), (8, 128), (16, 128)]:
    model_local = deepcopy(model)
    tome.patch_separate.dinov3(model_local, trace_source=False, prop_attn=True)
    model_local.r = [r] * len(model.backbone.blocks)
    evaluate(model_local, num_samples=num_samples, prefix=f'ToMe qk (r={r}):', batch_size=batch_size)
    del model_local
    torch.cuda.empty_cache()

# ToMe before block and unmerge after block
for r in [32, 64, 128, 256, 512]:
    model_local = deepcopy(model)
    tome.patch_block.dinov3(model_local)
    model_local.r = r
    evaluate(model_local, num_samples=num_samples, prefix=f'Block ToMe  (r={model_local.r}):', batch_size=batch_size)
    del model_local
    torch.cuda.empty_cache()

# Spatial ToMe
for r in [32, 64, 128, 256, 512]:
    model_local = deepcopy(model)
    spatial_tome.patch_block.dinov3(model_local)
    model_local.r = r
    evaluate(model_local, num_samples=num_samples, prefix=f'Spatial ToMe  (r={model_local.r}):', batch_size=batch_size)
    del model_local
    torch.cuda.empty_cache()

No ToMe:                 Accuracy: 81.43%,	FLOPs: 14513.7 GFLOPs,	Time: 182.8 ms
ToMe (r=4):              Accuracy: 79.95%,	FLOPs: 13337.1 GFLOPs,	Time: 187.6 ms
ToMe (r=8):              Accuracy: 80.68%,	FLOPs: 12168.4 GFLOPs,	Time: 170.0 ms
ToMe (r=16):             Accuracy: 79.32%,	FLOPs:  9863.4 GFLOPs,	Time: 141.1 ms
ToMe (r=32):             Accuracy: 33.23%,	FLOPs:  5744.9 GFLOPs,	Time:  99.5 ms
ToMe Pre (r=4):          Accuracy: 80.44%,	FLOPs: 13327.2 GFLOPs,	Time: 201.9 ms
ToMe Pre (r=8):          Accuracy: 80.47%,	FLOPs: 12149.0 GFLOPs,	Time: 181.3 ms
ToMe Pre (r=16):         Accuracy: 79.24%,	FLOPs:  9827.4 GFLOPs,	Time: 151.7 ms
ToMe Pre (r=32):         Accuracy: 26.48%,	FLOPs:  5693.4 GFLOPs,	Time: 108.6 ms
ToMe qk (r=(32, 32)):    Accuracy: 30.31%,	FLOPs:  5694.2 GFLOPs,	Time: 110.6 ms
ToMe qk (r=(8, 64)):     Accuracy: 80.22%,	FLOPs: 12119.2 GFLOPs,	Time: 190.1 ms
ToMe qk (r=(16, 64)):    Accuracy: 77.35%,	FLOPs:  9806.8 GFLOPs,	Time: 159.8 ms
ToMe qk (r=(8, 128)):    Acc